# Glacier grids from RGI:

Creates monthly grid files for the MBM to make PMB predictions over the whole glacier grid. The files come from the RGI grid with OGGM topography. Computing takes a long time because of the conversion to monthly format.
## Setting up:

In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
# --- System & utilities ---
import os
import sys
import warnings
from tqdm.notebook import tqdm
import rioxarray
import pandas as pd
# Add repo root for MBM imports
sys.path.append(os.path.join(os.getcwd(), "../../"))
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from datetime import datetime
import glob
import xarray as xr
from concurrent.futures import ProcessPoolExecutor, as_completed
import glob

# --- Data science stack ---
import matplotlib.pyplot as plt

# --- Custom MBM modules ---
import massbalancemachine as mbm

# --- Warnings & autoreload (notebook) ---
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2


# --- Configuration ---
cfg = mbm.EuropeConfig()

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.oggm import *
from regions.TF_Europe.scripts.geodata import *

# Plot styles:
mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

print("Using seed:", cfg.seed)

if torch.cuda.is_available():
    print("CUDA is available")
    mbm.utils.free_up_cuda()
else:
    print("CUDA is NOT available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# # Read rgis for all stakes:
# rgi_path = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_all_regions.csv")
# rgi_per_region = pd.read_csv(rgi_path, dtype={"RGI_REGION": str})

# # concatenate with HMA:
# # Read rgis for all stakes:
# rgi_path_HMA = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_HMA.csv")
# rgi_ids_HMA = pd.read_csv(rgi_path_HMA, dtype={"RGI_REGION": str})
# rgi_per_region = pd.concat([rgi_per_region, rgi_ids_HMA], ignore_index=True)

# rgi_per_region = rgi_per_region.dropna(subset=['RGIId', 'RGI_REGION']) \
#                            .groupby('RGI_REGION')['RGIId'] \
#                            .unique() \
#                            .to_dict()

## Create RGI grids for all glaciers:

### Create masked xarray grids: 
#### Check resolution:
(need to have computed svf separately for this)

In [ ]:
import xarray as xr
import numpy as np
import glob, os
import pandas as pd
from tqdm.notebook import tqdm


def get_region_grid_stats(cfg, region_id):
    path_RGIs = os.path.join(cfg.dataPath, f"OGGM/rgi_region_{region_id}",
                             "xr_grids")
    zarr_paths = sorted(glob.glob(os.path.join(path_RGIs, "*.zarr")))
    records = []
    for p in tqdm(zarr_paths, desc=f"RGI {region_id}", unit="glacier"):
        rgi_id = os.path.splitext(os.path.basename(p))[0]
        try:
            ds = xr.open_zarr(p)
            dx = float(np.abs(np.diff(ds["x"].values)).mean())
            dy = float(np.abs(np.diff(ds["y"].values)).mean())
            nx, ny = ds.sizes.get("x"), ds.sizes.get("y")
            records.append((rgi_id, dx, dy, nx, ny, nx * dx, ny * dy, None))
            ds.close()
        except Exception as e:
            records.append((rgi_id, None, None, None, None, None, None,
                            f"{type(e).__name__}: {e}"))

    df = pd.DataFrame(records,
                      columns=[
                          "rgi_id", "dx_m", "dy_m", "nx", "ny", "extent_x_m",
                          "extent_y_m", "error"
                      ])
    df["max_extent_m"] = df[["extent_x_m", "extent_y_m"]].max(axis=1)
    df["n_pixels"] = df["nx"] * df["ny"]
    df["region"] = region_id
    return df


regions_to_check = ["01", "11", "13"]  # Alaska, Central Europe, Central Asia
all_dfs = {}
for r in tqdm(regions_to_check, desc="Regions", unit="region"):
    all_dfs[r] = get_region_grid_stats(cfg, r)

for r, df in all_dfs.items():
    n_err = df["error"].notna().sum()
    print(f"\n=== RGI {r} ===  ({n_err} failed to open)")
    if n_err:
        print(df.loc[df["error"].notna(), ["rgi_id", "error"]].head(10))
    print(df[["dx_m", "max_extent_m",
              "n_pixels"]].describe(percentiles=[.25, .5, .75, .9, .99]))
    print("Top 5 largest by extent:")
    print(
        df.nlargest(
            5, "max_extent_m")[["rgi_id", "dx_m", "max_extent_m", "n_pixels"]])

#### Masked grids:

In [ ]:
def get_target_resolution(ds, dx_m, dy_m):
    """
    Pick a target resolution based on glacier physical extent, not native dx_m
    (since OGGM's own dx_m caps around 200m regardless of glacier size).
    """
    nx = ds.sizes.get("x")
    ny = ds.sizes.get("y")
    extent_x_m = nx * dx_m
    extent_y_m = ny * dy_m
    max_extent_m = max(extent_x_m, extent_y_m)

    if max_extent_m < 500:
        return None  # keep native resolution
    elif max_extent_m < 5_000:
        return 50
    elif max_extent_m < 20_000:
        return 100
    else:
        return 200

In [ ]:
def process_one_glacier(
    rgi_gl: str,
    path_RGIs: str,
    path_xr_svf: str,
    path_xr_grids: str,
    target_res_m: int = 50,
):
    """
    Worker: load OGGM grid, mask, optional coarsen, reproject to lat/lon,
    merge SVF, write per-glacier zarr. Returns a small status tuple.
    """
    try:
        # 1) Masked OGGM grid in projected coords
        ds, _ = create_masked_glacier_grid(path_RGIs, rgi_gl)

        # 2) Optional coarsen in projected space, based on glacier extent
        dx_m, dy_m = get_res_from_projected(ds)
        chosen_target_res_m = target_res_m or get_target_resolution(
            ds, dx_m, dy_m)

        if chosen_target_res_m is not None and dx_m < chosen_target_res_m:
            ds = coarsenDS_mercator(ds, target_res_m=chosen_target_res_m)

        # 3) Reproject to WGS84 lat/lon
        original_proj = ds.pyproj_srs
        ds = ds.rio.write_crs(original_proj)
        ds_latlon = ds.rio.reproject("EPSG:4326").rename({
            "x": "lon",
            "y": "lat"
        })


        # 4) Load SVF + merge (if exists)
        svf_path = os.path.join(path_xr_svf, f"{rgi_gl}_svf_latlon.nc")
        if os.path.exists(svf_path):
            with xr.open_dataset(svf_path) as ds_svf:
                ds_svf = ds_svf.load(
                )  # optional: helps avoid file-handle issues in multiprocessing

                # Normalize coord names
                if "x" in ds_svf.dims or "y" in ds_svf.dims:
                    ds_svf = ds_svf.rename({"x": "lon", "y": "lat"})
                if "longitude" in ds_svf.dims or "latitude" in ds_svf.dims:
                    ds_svf = ds_svf.rename({
                        "longitude": "lon",
                        "latitude": "lat"
                    })

            # Sort ascending for interp stability
            if ds_latlon.lon[0] > ds_latlon.lon[-1]:
                ds_latlon = ds_latlon.sortby("lon")
            if ds_latlon.lat[0] > ds_latlon.lat[-1]:
                ds_latlon = ds_latlon.sortby("lat")
            if ds_svf.lon[0] > ds_svf.lon[-1]:
                ds_svf = ds_svf.sortby("lon")
            if ds_svf.lat[0] > ds_svf.lat[-1]:
                ds_svf = ds_svf.sortby("lat")

            svf_vars = [
                v for v in ("svf", "asvf", "opns") if v in ds_svf.data_vars
            ]

            if svf_vars:
                # Merge directly if grids match; else interpolate
                if (np.array_equal(ds_latlon.lon.values, ds_svf.lon.values)
                        and np.array_equal(ds_latlon.lat.values,
                                           ds_svf.lat.values)):
                    ds_latlon = xr.merge([ds_latlon, ds_svf[svf_vars]])
                else:
                    svf_on_grid = ds_svf[svf_vars].interp(lon=ds_latlon.lon,
                                                          lat=ds_latlon.lat,
                                                          method="linear")
                    for v in svf_vars:
                        svf_on_grid[v] = svf_on_grid[v].astype("float32")
                    ds_latlon = ds_latlon.assign(
                        **{v: svf_on_grid[v]
                           for v in svf_vars})

                # Masked SVF versions using glacier_mask (if present)
                if "glacier_mask" in ds_latlon:
                    gmask = xr.where(
                        ds_latlon["glacier_mask"].values.astype(bool), 1.0,
                        np.nan)
                    for v in svf_vars:
                        ds_latlon[f"masked_{v}"] = gmask * ds_latlon[v]

        # 5) Save final lat/lon grid
        save_path = os.path.join(path_xr_grids, f"{rgi_gl}.zarr")
        ds_latlon.to_zarr(save_path, mode="w")

        return (rgi_gl, "ok", "")

    except Exception as e:
        return (rgi_gl, "error", f"{type(e).__name__}: {e}")


def glacier_ids_from_xr_grids(path_RGIs: str):
    """
    If you have per-glacier OGGM zarr grids saved as RGI60-..xxxx.zarr
    in path_RGIs, return those ids.
    """
    zarrs = sorted(glob.glob(os.path.join(path_RGIs, "*.zarr")))
    return [os.path.splitext(os.path.basename(p))[0]
            for p in zarrs]  # stem before .zarr


def run_parallel_processing_region(
    rgi_ids,
    path_RGIs,
    path_xr_svf,
    path_xr_grids,
    n_workers=6,
    clear_out=False,
    target_res_m=50,
):
    if clear_out:
        emptyfolder(path_xr_grids)
    else:
        os.makedirs(path_xr_grids, exist_ok=True)

    # your existing executor code, but using rgi_ids directly:
    results = []
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {
            ex.submit(
                process_one_glacier,
                rgi_id,
                path_RGIs,
                path_xr_svf,
                path_xr_grids,
                target_res_m,
            ):
            rgi_id
            for rgi_id in rgi_ids
        }

        for fut in tqdm(as_completed(futures), total=len(futures)):
            results.append(fut.result())

    n_ok = sum(r[1] == "ok" for r in results)
    n_err = sum(r[1] == "error" for r in results)
    print(f"Done. ok={n_ok}, error={n_err}")

    if n_err:
        for rgi_id, status, msg in results:
            if status == "error":
                print(f"[{rgi_id}] {msg}")

    return results


def run_all_regions(
    cfg,
    RGI_REGIONS,
    rgi_per_region=None,
    stake_only_regions=None,  # e.g. {"01", "02", "07"} — regions to filter by stake RGIs
    n_workers=6,
    clear_out=False,
    target_res_m=50,
):
    all_results = {}

    for rgi_region, spec in RGI_REGIONS.items():
        rgi_region_str = str(rgi_region).zfill(2)
        region_folder = spec["folder"]
        print(f"\n========== RGI {rgi_region_str} ({spec['name']}) ==========")

        path_xr_svf = os.path.join(cfg.dataPath, "RGI_v6", region_folder,
                                   "svf_nc_latlon")
        path_xr_grids_out = os.path.join(cfg.dataPath, "RGI_v6", region_folder,
                                         "xr_masked_grids")
        path_RGIs = os.path.join(cfg.dataPath,
                                 f"OGGM/rgi_region_{rgi_region_str}",
                                 "xr_grids")

        if not os.path.isdir(path_RGIs):
            print(f"Skipping: missing xr_grids folder: {path_RGIs}")
            continue
        if not os.path.isdir(path_xr_svf):
            print(f"Skipping: missing svf folder: {path_xr_svf}")
            continue

        rgi_ids = glacier_ids_from_xr_grids(path_RGIs)
        print(f"Found {len(rgi_ids)} glacier grids in {path_RGIs}")

        # Filter to stake RGIs if this region is in stake_only_regions
        if stake_only_regions and rgi_region_str in stake_only_regions:
            if rgi_per_region and rgi_region_str in rgi_per_region:
                allowed = set(rgi_per_region[rgi_region_str])
                rgi_ids = [r for r in rgi_ids if r in allowed]
                print(f"Filtered to {len(rgi_ids)} stake RGIs.")
            else:
                print(
                    f"⚠️  stake_only requested for {rgi_region_str} but no rgi_per_region entry found, using all."
                )

        if len(rgi_ids) == 0:
            continue

        os.makedirs(path_xr_grids_out, exist_ok=True)
        results = run_parallel_processing_region(
            rgi_ids=rgi_ids,
            path_RGIs=path_RGIs,
            path_xr_svf=path_xr_svf,
            path_xr_grids=path_xr_grids_out,
            n_workers=n_workers,
            clear_out=clear_out,
            target_res_m=target_res_m,
        )
        all_results[rgi_region] = results

    return all_results

In [ ]:
RUN = False

if RUN:
    all_results = run_all_regions(
        cfg=cfg,
        RGI_REGIONS=RGI_REGIONS,
        rgi_per_region=None,  # dict of region -> array of RGIIds
        stake_only_regions={},  # only filter these regions
        n_workers=6,
        clear_out=False,
        target_res_m=None,  # None = auto-pick per glacier based on extent; set e.g. 50 to force one resolution for all
    )

In [ ]:
# Look at an example:
rgi_id = "RGI60-07.00246"
region_id = "07"
# --- Paths ---
basepath = os.path.join(cfg.dataPath, "RGI_v6",
                        RGI_REGIONS[region_id]["folder"])
dem_path = os.path.join(basepath, "geotiff", f"{rgi_id}.tif")
zarr_path = os.path.join(basepath, "xr_masked_grids", f"{rgi_id}.zarr")
svf_path = os.path.join(basepath, "svf_nc_latlon", f"{rgi_id}_svf_latlon.nc")

# --- Load data ---
dem = rioxarray.open_rasterio(dem_path).squeeze()
ds = xr.open_zarr(zarr_path)
ds_svf = xr.open_dataset(svf_path)

# Handle coord naming for SVF
if "lon" not in ds_svf.coords:
    ds_svf = ds_svf.rename({"x": "lon", "y": "lat"})

# --- Figure layout ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

# 1️⃣ DEM (projected)
dem.plot(ax=axes[0], cmap="terrain")
axes[0].set_title("DEM (projected meters)")
axes[0].set_xlabel("Easting [m]")
axes[0].set_ylabel("Northing [m]")

# 2️⃣ Masked aspect (projected OGGM grid)
ds["masked_aspect"].plot(ax=axes[1])
axes[1].set_title("Masked Aspect (°)")
axes[1].set_xlabel("Longitude (°)")
axes[1].set_ylabel("Latitude (°)")

# 3️⃣ SVF (lat/lon)
ds["svf"].plot(ax=axes[2])

axes[2].set_title("Sky View Factor (lat/lon)")
axes[2].set_xlabel("Longitude (°)")
axes[2].set_ylabel("Latitude (°)")

plt.suptitle(f"{rgi_id}", fontsize=15)
plt.show()

## Create monthly grids:

In [ ]:
vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
    "era5_climate_data_HMA":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data_HMA':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

In [ ]:
import xarray as xr
import numpy as np
import glob, os
import pandas as pd
from tqdm.notebook import tqdm

def get_region_grid_stats(cfg, region_id):
    path_RGIs = os.path.join(cfg.dataPath, f"OGGM/rgi_region_{region_id}", "xr_grids")
    zarr_paths = sorted(glob.glob(os.path.join(path_RGIs, "*.zarr")))
    records = []
    for p in tqdm(zarr_paths, desc=f"RGI {region_id}", unit="glacier"):
        rgi_id = os.path.splitext(os.path.basename(p))[0]
        try:
            ds = xr.open_zarr(p)
            dx = float(np.abs(np.diff(ds["x"].values)).mean())
            dy = float(np.abs(np.diff(ds["y"].values)).mean())
            nx, ny = ds.sizes.get("x"), ds.sizes.get("y")

            # cheap corner-only reprojection to get lat/lon bounds
            ds_crs = ds.rio.write_crs(ds.pyproj_srs)
            lon_min, lat_min, lon_max, lat_max = ds_crs.rio.transform_bounds("EPSG:4326")

            records.append((rgi_id, dx, dy, nx, ny, nx * dx, ny * dy,
                            lon_min, lon_max, lat_min, lat_max, None))
            ds.close()
        except Exception as e:
            records.append((rgi_id, None, None, None, None, None, None,
                            None, None, None, None, f"{type(e).__name__}: {e}"))

    df = pd.DataFrame(records,
                      columns=["rgi_id", "dx_m", "dy_m", "nx", "ny",
                               "extent_x_m", "extent_y_m",
                               "lon_min", "lon_max", "lat_min", "lat_max", "error"])
    df["max_extent_m"] = df[["extent_x_m", "extent_y_m"]].max(axis=1)
    df["n_pixels"] = df["nx"] * df["ny"]
    df["region"] = region_id
    return df

def compute_region_bbox(df, buffer_deg=0.5):
    """Empirical bbox from actual glacier corner coordinates, with a small buffer."""
    valid = df.dropna(subset=["lon_min", "lon_max", "lat_min", "lat_max"])
    return {
        "lat": (float(valid["lat_min"].min()) - buffer_deg,
                float(valid["lat_max"].max()) + buffer_deg),
        "lon": (float(valid["lon_min"].min()) - buffer_deg,
                float(valid["lon_max"].max()) + buffer_deg),
    }

regions_to_check = ["01", "06", "07", "08", "11", "13"]
all_dfs = {}
for r in tqdm(regions_to_check, desc="Regions", unit="region"):
    all_dfs[r] = get_region_grid_stats(cfg, r)


RGI_ERA5_SOURCE = {
    "01": "EU_US_CANADA",
    "06": "EU_US_CANADA",
    "07": "EU_US_CANADA",
    "08": "EU_US_CANADA",
    "11": "EU_US_CANADA",
    "13": "HMA",
}

RGI_BBOX_EMPIRICAL = {}
for r, df in all_dfs.items():
    n_err = df["error"].notna().sum()
    bbox = compute_region_bbox(df)
    RGI_BBOX_EMPIRICAL[r] = bbox
    print(f"RGI {r}: ({n_err} failed to open) -> bbox = {bbox}")

print("\nRGI_BBOX = {")
for r, bbox in RGI_BBOX_EMPIRICAL.items():
    print(f'    "{r}": {{"lat": {bbox["lat"]}, "lon": {bbox["lon"]}}},')
print("}")

In [ ]:
RGI_BBOX = RGI_BBOX_EMPIRICAL

In [ ]:
# RGI_BBOX = {
#     "01": {
#         "lat": (52, 72),
#         "lon": (-172, -128)
#     },  # Alaska
#     "06": {
#         "lat": (62, 67),
#         "lon": (-25, -12)
#     },  # Iceland
#     "07": {
#         "lat": (74, 82),
#         "lon": (8, 36)
#     },  # Svalbard
#     "08": {
#         "lat": (56, 72),
#         "lon": (4, 32)
#     },  # Scandinavia
#     "11": {
#         "lat": (41, 49),
#         "lon": (5, 17)
#     },  # Central Europe
#     "13": {
#         "lat": (25, 45),
#         "lon": (60, 105)
#     },  # Central Asia
# }

In [ ]:
def compute_glacier_areas_km2(zarr_dir, rgi_ids, cache_path=None, force_recompute=False):
    """
    Compute true glacier area (km²) from glacier_mask pixel count × pixel area,
    using the already-reprojected lat/lon masked grids in zarr_dir.
    Caches results to avoid recomputing on every run.
    """
    if cache_path and os.path.exists(cache_path) and not force_recompute:
        print(f"  Loading cached areas: {cache_path}")
        return pd.read_csv(cache_path)

    records = []
    for rgi_id in tqdm(rgi_ids, desc="Computing areas", unit="glacier"):
        try:
            ds = xr.open_zarr(os.path.join(zarr_dir, f"{rgi_id}.zarr"))
            gl_mask = ds["glacier_mask"].values.astype(bool)
            n_glacier_px = int(gl_mask.sum())

            # approximate pixel area in m² from lat/lon spacing at the glacier's mean latitude
            lat_res = float(np.abs(np.diff(ds["lat"].values)).mean())
            lon_res = float(np.abs(np.diff(ds["lon"].values)).mean())
            mean_lat = float(ds["lat"].values.mean())
            m_per_deg_lat = 111_320
            m_per_deg_lon = 111_320 * np.cos(np.deg2rad(mean_lat))
            px_area_m2 = (lat_res * m_per_deg_lat) * (lon_res * m_per_deg_lon)

            area_km2 = n_glacier_px * px_area_m2 / 1e6
            records.append((rgi_id, area_km2, None))
            ds.close()
        except Exception as e:
            records.append((rgi_id, None, f"{type(e).__name__}: {e}"))

    df = pd.DataFrame(records, columns=["rgi_id", "area_km2", "error"])
    if cache_path:
        df.to_csv(cache_path, index=False)
        print(f"  Cached areas: {cache_path}")
    return df

In [ ]:
def glacier_ids_from_xr_grids(path):
    zarrs = sorted(glob.glob(os.path.join(path, "*.zarr")))
    return [os.path.splitext(os.path.basename(p))[0] for p in zarrs]


def clip_era5_for_region(era5_path, geopot_path, region_id, out_dir):
    """Clip ERA5 files to region bounding box. Skips if already exists."""
    bbox = RGI_BBOX[region_id]
    os.makedirs(out_dir, exist_ok=True)

    clipped = {}
    for path, name in [(era5_path, "era5"), (geopot_path, "geopot")]:
        out_path = os.path.join(out_dir, f"{name}_{region_id}.nc")
        if os.path.exists(out_path):
            print(f"  Already clipped: {out_path}")
        else:
            print(f"  Clipping {name} for region {region_id}...")
            ds = xr.open_dataset(path)
            ds.sel(
                latitude=slice(bbox["lat"][1],
                               bbox["lat"][0]),  # ERA5 lat is descending
                longitude=slice(bbox["lon"][0], bbox["lon"][1]),
            ).to_netcdf(out_path)
            print(
                f"  Saved ({os.path.getsize(out_path)/1e9:.2f} GB): {out_path}"
            )
        clipped[name] = out_path

    return {
        "era5_climate_data": clipped["era5"],
        "geopotential_data": clipped["geopot"],
    }


def check_glaciers_in_bbox(rgi_ids, zarr_dir, bbox):
    """Warn if any glacier centroids fall outside the clipped ERA5 bbox."""
    outside = []
    for rgi_id in rgi_ids:
        try:
            ds = xr.open_zarr(os.path.join(zarr_dir, f"{rgi_id}.zarr"))
            gl_mask = ds["glacier_mask"].values.astype(bool)
            lon_2d, lat_2d = np.meshgrid(ds["lon"].values, ds["lat"].values)
            mean_lon = float(lon_2d[gl_mask].mean())
            mean_lat = float(lat_2d[gl_mask].mean())
            if not (bbox["lat"][0] <= mean_lat <= bbox["lat"][1]
                    and bbox["lon"][0] <= mean_lon <= bbox["lon"][1]):
                outside.append((rgi_id, mean_lat, mean_lon))
        except Exception as e:
            print(f"  Could not check {rgi_id}: {e}")
    return outside


def run_one_region(
    cfg,
    region_id,
    spec,
    rgi_per_region,
    paths,
    vois_climate,
    vois_topographical,
    meta_cols,
    years=range(2010, 2025),
    start_month="10",
    n_workers=6,
    force_recompute=False,
    min_area_km2=1.0
):
    """Process all glaciers for a single RGI region."""
    region_id_str = str(region_id).zfill(2)
    print(f"\n========== RGI {region_id_str} ({spec['name']}) ==========")

    basepath = os.path.join(cfg.dataPath, "RGI_v6", spec["folder"])
    zarr_dir = os.path.join(basepath, "xr_masked_grids")
    out_dir = os.path.join(basepath, "monthly_parquets")
    os.makedirs(out_dir, exist_ok=True)

    # --- optionally wipe existing parquets ---
    if force_recompute:
        existing = glob.glob(os.path.join(out_dir, "**", "*.parquet"),
                             recursive=True)
        if existing:
            print(
                f"  force_recompute=True: deleting {len(existing)} existing parquet files..."
            )
            for f in existing:
                os.remove(f)
        else:
            print(f"  force_recompute=True: no existing parquet files found.")

    if not os.path.isdir(zarr_dir):
        print(f"  Skipping: missing xr_masked_grids: {zarr_dir}")
        return {"ok": 0, "skip": 0, "err": 0}

    # --- resolve ERA5 source files for this region ---
    era5_source = RGI_ERA5_SOURCE.get(region_id_str, "EU_US_CANADA")
    suffix = f"_{era5_source}" if era5_source != "EU_US_CANADA" else ""
    base_paths = {
        "era5_climate_data": paths[f"era5_climate_data{suffix}"],
        "geopotential_data": paths[f"geopotential_data{suffix}"],
    }
    print(f"  Using ERA5 source: {era5_source}")

    # --- clip ERA5 to this region ---
    era5_clip_dir = os.path.join(cfg.dataPath, "ERA5Land/clipped_to_RGI")
    if region_id_str in RGI_BBOX:
        region_paths = clip_era5_for_region(
            base_paths["era5_climate_data"],
            base_paths["geopotential_data"],
            region_id_str,
            out_dir=era5_clip_dir,
        )
    else:
        print(
            f"  No bounding box defined for region {region_id_str}, using full ERA5."
        )
        region_paths = base_paths

    # --- build task list ---
    rgi_ids = glacier_ids_from_xr_grids(zarr_dir)

    # --- NEW: filter by glacier area ---
    if min_area_km2 is not None:
        area_cache = os.path.join(basepath, f"glacier_areas_km2.csv")
        areas_df = compute_glacier_areas_km2(zarr_dir, rgi_ids, cache_path=area_cache)
        valid_areas = areas_df.dropna(subset=["area_km2"])
        kept = set(valid_areas.loc[valid_areas["area_km2"] >= min_area_km2, "rgi_id"])
        n_before = len(rgi_ids)
        rgi_ids = [r for r in rgi_ids if r in kept]
        print(f"  Area filter (>= {min_area_km2} km²): {n_before} -> {len(rgi_ids)} glaciers")

    # --- sanity check: glaciers inside bbox ---
    if region_id_str in RGI_BBOX:
        outside = check_glaciers_in_bbox(rgi_ids, zarr_dir, RGI_BBOX[region_id_str])
        if outside:
            print(f"  ⚠️  {len(outside)} glaciers outside clipped bbox:")
            for rgi_id, lat, lon in outside:
                print(f"    {rgi_id}: lat={lat:.2f}, lon={lon:.2f}")
        else:
            print(f"  ✓ All {len(rgi_ids)} glaciers inside clipped bbox")

    tasks = [(r, y) for r in rgi_ids for y in years]
    print(f"  {len(rgi_ids)} glaciers × {len(list(years))} years = {len(tasks)} tasks")
    if not tasks:
        return {"ok": 0, "skip": 0, "err": 0}

    # --- logging ---
    n_ok = n_err = n_skip = 0
    log_dir = Path("logs/errors")
    log_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = log_dir / f"rgi_{region_id_str}_{ts}.log"
    err_path = log_dir / f"rgi_{region_id_str}_{ts}_ERRORS.log"

    with open(log_path, "w") as f:
        f.write(
            f"Run started: {datetime.now().isoformat(timespec='seconds')}\n")
        f.write(f"max_workers={n_workers}, n_tasks={len(tasks)}\n")
        f.write("-" * 80 + "\n")

    # --- parallel execution ---
    with ProcessPoolExecutor(
            max_workers=n_workers,
            initializer=grid_inputs_rgi.init_worker_rgi,
    ) as ex:
        futs = [
            ex.submit(
                grid_inputs_rgi.process_monthly_grids_rgi,
                region_id_str,
                rgi_id,
                year,
                zarr_path=os.path.join(zarr_dir, f"{rgi_id}.zarr"),
                basepath=basepath,
                out_folder_root=out_dir,
                vois_climate=vois_climate,
                vois_topo=vois_topographical,
                meta_cols=meta_cols,
                era5_monthly_path=region_paths["era5_climate_data"],
                era5_geopot_path=region_paths["geopotential_data"],
                start_month=start_month,
                cfg=cfg,
            ) for rgi_id, year in tasks
        ]

        with tqdm(total=len(futs), desc=f"RGI {region_id_str}",
                  unit="task") as pbar:
            for fut in as_completed(futs):
                try:
                    res = fut.result()
                except Exception as e:
                    res = f"ERROR <future>: {type(e).__name__}: {e}"

                if res.startswith("OK"): n_ok += 1
                elif res.startswith("ERROR"): n_err += 1
                elif res.startswith("SKIP"): n_skip += 1

                if res.startswith(("SKIP", "ERROR")):
                    line = f"[{datetime.now().isoformat(timespec='seconds')}] {res}\n"
                    with open(log_path, "a") as f:
                        f.write(line)
                    if res.startswith("ERROR"):
                        with open(err_path, "a") as f:
                            f.write(line)

                pbar.set_postfix(ok=n_ok, skip=n_skip, err=n_err)
                pbar.update(1)

    with open(log_path, "a") as f:
        f.write("-" * 80 + "\n")
        f.write(
            f"Run finished: {datetime.now().isoformat(timespec='seconds')}\n")
        f.write(f"OK={n_ok}, SKIP={n_skip}, ERROR={n_err}\n")

    print(f"  Done. OK={n_ok}, SKIP={n_skip}, ERROR={n_err}")
    if n_err: print(f"  Errors: {err_path}")
    # --- free memory after region is done ---
    import gc
    gc.collect()

    return {"ok": n_ok, "skip": n_skip, "err": n_err}


def run_all_glacier_grids(
    cfg, RGI_REGIONS, rgi_per_region, paths, vois_climate, vois_topographical,
    meta_cols, years=range(2010, 2025), start_month="10", n_workers=6,
    force_recompute=False, min_area_km2=1.0,  # NEW
):
    return {
        region_id: run_one_region(
            cfg=cfg, region_id=region_id, spec=spec, rgi_per_region=rgi_per_region,
            paths=paths, vois_climate=vois_climate, vois_topographical=vois_topographical,
            meta_cols=meta_cols, years=years, start_month=start_month,
            n_workers=n_workers, force_recompute=force_recompute, min_area_km2=min_area_km2,
        )
        for region_id, spec in RGI_REGIONS.items()
    }

In [ ]:
META_COLS = [
    'RGIId', 'POINT_ID', 'ID', 'GLWD_ID', 'N_MONTHS', 'MONTHS', 'PERIOD',
    'GLACIER'
]

RUN = True
max_workers = max(1, (os.cpu_count() or 4) - 1)

RGI_REGIONS_TEST = {
    '11': {
        'name': 'CentralEurope',
        'folder': 'RGI_11_CentralEurope',
        'file': '11_rgi60_CentralEurope.shp',
        'code': 'CEU',
        'subregions': ['France', 'Switzerland', 'Italy_Austria'],
        'subregions_codes': ['FR', 'CH', 'IT_AT'],
        'era5_source': 'EU_US_CANADA'
    },
}

if RUN:
    all_results = run_all_glacier_grids(cfg=cfg,
                                        RGI_REGIONS=RGI_REGIONS_TEST,
                                        rgi_per_region=None,
                                        paths=paths,
                                        vois_climate=vois_climate,
                                        vois_topographical=vois_topographical,
                                        meta_cols=META_COLS,
                                        years=range(2000, 2025),
                                        start_month="10",
                                        n_workers=10,
                                        force_recompute=True)

In [ ]:
META_COLS = [
    'RGIId', 'POINT_ID', 'ID', 'GLWD_ID', 'N_MONTHS', 'MONTHS', 'PERIOD',
    'GLACIER'
]

RUN = True
max_workers = max(1, (os.cpu_count() or 4) - 1)

RGI_REGIONS = {
    '06': {
        'name': 'Iceland',
        'folder': 'RGI_06_Iceland',
        'file': '06_rgi60_Iceland.shp',
        'code': 'ISL',
        'era5_source': 'EU_US_CANADA'
    },
    '07': {
        'name': 'Svalbard',
        'folder': 'RGI_07_Svalbard',
        'file': '07_rgi60_Svalbard.shp',
        'code': 'SJM',
        'era5_source': 'EU_US_CANADA'
    },
    '08': {
        'name': 'Scandinavia',
        'folder': 'RGI_08_Scandinavia',
        'file': '08_rgi60_Scandinavia.shp',
        'code': 'SCA',
        'subregions': ['Norway'],
        'subregions_codes': ['NOR'],
        'era5_source': 'EU_US_CANADA'
    },
    '11': {
        'name': 'CentralEurope',
        'folder': 'RGI_11_CentralEurope',
        'file': '11_rgi60_CentralEurope.shp',
        'code': 'CEU',
        'subregions': ['France', 'Switzerland', 'Italy_Austria'],
        'subregions_codes': ['FR', 'CH', 'IT_AT'],
        'era5_source': 'EU_US_CANADA'
    },
}

if RUN:
    all_results = run_all_glacier_grids(cfg=cfg,
                                        RGI_REGIONS=RGI_REGIONS,
                                        rgi_per_region=None,
                                        paths=paths,
                                        vois_climate=vois_climate,
                                        vois_topographical=vois_topographical,
                                        meta_cols=META_COLS,
                                        years=range(2000, 2025),
                                        start_month="10",
                                        n_workers=10,
                                        force_recompute=False)